In [1]:
import numpy as np
import json
import pandas as pd

In [2]:
def seed_everything(seed: int):
    import random, os
    import numpy as np
    import torch

    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

In [3]:
seed_everything(42)

In [4]:
with open(f'./coqa-dev-v1.0.json', 'r') as infile:
    data = json.load(infile)['data']

dataset = {}

dataset['story'] = []
dataset['question'] = []
dataset['answer'] = []
dataset['additional_answers'] = []
dataset['id'] = []

for sample_id, sample in enumerate(data):
    story = sample['story']
    questions = sample['questions']
    answers = sample['answers']
    additional_answers = sample['additional_answers']
    for question_index, question in enumerate(questions):
        dataset['story'].append(story)
        dataset['question'].append(question['input_text'])
        dataset['answer'].append({
            'text': answers[question_index]['input_text'],
            'answer_start': answers[question_index]['span_start']
        })
        dataset['id'].append(sample['id'] + '_' + str(question_index))
        additional_answers_list = []

        for i in range(3):
            additional_answers_list.append(additional_answers[str(i)][question_index]['input_text'])

        dataset['additional_answers'].append(additional_answers_list)
        story = story + ' Q: ' + question['input_text'] + ' A: ' + answers[question_index]['input_text']
        if not story[-1] == '.':
            story = story + '.'

dataset_df = pd.DataFrame.from_dict(dataset)

In [5]:
dataset_df

,story,question,answer,additional_answers,id
0,"Once upon a time, in a barn near a farm house,...",What color was Cotton?,"{'text': 'white', 'answer_start': 59}","[white, white, white]",3dr23u6we5exclen4th8uq9rb42tel_0
1,"Once upon a time, in a barn near a farm house,...",Where did she live?,"{'text': 'in a barn', 'answer_start': 18}","[in a barn, in a barn, in a barn near]",3dr23u6we5exclen4th8uq9rb42tel_1
2,"Once upon a time, in a barn near a farm house,...",She live alone?,"{'text': 'no', 'answer_start': 196}","[no, No, no]",3dr23u6we5exclen4th8uq9rb42tel_2
3,"Once upon a time, in a barn near a farm house,...",Who did she live with?,"{'text': 'with her mommy and 5 sisters', 'answ...","[her mommy and 5 other sisters, her mommy and ...",3dr23u6we5exclen4th8uq9rb42tel_3
4,"Once upon a time, in a barn near a farm house,...",What color was her sisters?,"{'text': 'orange and white', 'answer_start': 428}","[orange with white tiger stripes, orange, orange]",3dr23u6we5exclen4th8uq9rb42tel_4
...,...,...,...,...,...
7978,"Las Vegas (, Spanish for ""The Meadows""), offic...","where do the nickname ""Sin City"" come from?",{'text': 'The city's tolerance for numerous fo...,[tolerance for numerous forms of adult enterta...,3ixqg4fa2tygl3tpwwa12i2uf58b90_7
7979,"Las Vegas (, Spanish for ""The Meadows""), offic...",Which state it in?,"{'text': 'Nevada', 'answer_start': 100}","[Nevada, Nevada,, Nevada]",3ixqg4fa2tygl3tpwwa12i2uf58b90_8
7980,"Las Vegas (, Spanish for ""The Meadows""), offic...",Is it located in a desert?,"{'text': 'Yes', 'answer_start': 326}","[yes, yes, Yes]",3ixqg4fa2tygl3tpwwa12i2uf58b90_9
7981,"Las Vegas (, Spanish for ""The Meadows""), offic...",what that name of that desert?,"{'text': 'Mojave Desert.', 'answer_start': 345}","[Mojave Desert, Mojave Desert., Mojave]",3ixqg4fa2tygl3tpwwa12i2uf58b90_10


In [6]:
answers = []
for i in range(len(dataset_df)):
    answer = np.load(f"/app/haloscope/save_for_eval/coqa_hal_det/answers/most_likely_hal_det_llama2_chat_7B_coqa_answers_index_{i}.npy")
    answers.append(answer[0])

In [7]:
dataset_df["generated_answer"] = answers

In [8]:
dataset_df = dataset_df.rename(columns={"story": "context"})

In [9]:
dataset_df

,context,question,answer,additional_answers,id,generated_answer
0,"Once upon a time, in a barn near a farm house,...",What color was Cotton?,"{'text': 'white', 'answer_start': 59}","[white, white, white]",3dr23u6we5exclen4th8uq9rb42tel_0,Cotton was a white kitten.
1,"Once upon a time, in a barn near a farm house,...",Where did she live?,"{'text': 'in a barn', 'answer_start': 18}","[in a barn, in a barn, in a barn near]",3dr23u6we5exclen4th8uq9rb42tel_1,above the barn.
2,"Once upon a time, in a barn near a farm house,...",She live alone?,"{'text': 'no', 'answer_start': 196}","[no, No, no]",3dr23u6we5exclen4th8uq9rb42tel_2,no.
3,"Once upon a time, in a barn near a farm house,...",Who did she live with?,"{'text': 'with her mommy and 5 sisters', 'answ...","[her mommy and 5 other sisters, her mommy and ...",3dr23u6we5exclen4th8uq9rb42tel_3,her mommy and 5 other sisters.
4,"Once upon a time, in a barn near a farm house,...",What color was her sisters?,"{'text': 'orange and white', 'answer_start': 428}","[orange with white tiger stripes, orange, orange]",3dr23u6we5exclen4th8uq9rb42tel_4,orange with white tiger stripes.
...,...,...,...,...,...,...
7978,"Las Vegas (, Spanish for ""The Meadows""), offic...","where do the nickname ""Sin City"" come from?",{'text': 'The city's tolerance for numerous fo...,[tolerance for numerous forms of adult enterta...,3ixqg4fa2tygl3tpwwa12i2uf58b90_7,its tolerance for numerous forms of adult ente...
7979,"Las Vegas (, Spanish for ""The Meadows""), offic...",Which state it in?,"{'text': 'Nevada', 'answer_start': 100}","[Nevada, Nevada,, Nevada]",3ixqg4fa2tygl3tpwwa12i2uf58b90_8,Nevada.
7980,"Las Vegas (, Spanish for ""The Meadows""), offic...",Is it located in a desert?,"{'text': 'Yes', 'answer_start': 326}","[yes, yes, Yes]",3ixqg4fa2tygl3tpwwa12i2uf58b90_9,Yes.
7981,"Las Vegas (, Spanish for ""The Meadows""), offic...",what that name of that desert?,"{'text': 'Mojave Desert.', 'answer_start': 345}","[Mojave Desert, Mojave Desert., Mojave]",3ixqg4fa2tygl3tpwwa12i2uf58b90_10,Mojave Desert.


In [10]:
gts = np.load(f'./ml_coqa_bleurt_score.npy')
gts_bg = np.load(f'./bg_coqa_bleurt_score.npy')
thres = 0.5
gt_label = np.asarray(gts> thres, dtype=np.int32)
gt_label_bg = np.asarray(gts_bg > thres, dtype=np.int32)

In [11]:
length = len(dataset_df)
wild_ratio = 0.75

permuted_index = np.random.permutation(length)
wild_q_indices = permuted_index[:int(wild_ratio * length)] # train + val
# exclude validation samples.
wild_q_indices1 = wild_q_indices[:len(wild_q_indices) - 100] # train
wild_q_indices2 = wild_q_indices[len(wild_q_indices) - 100:] # val
gt_label_test = []
gt_label_wild = []
gt_label_val = []
for i in range(length):
    if i not in wild_q_indices:
        gt_label_test.extend(gt_label[i: i+1])
    elif i in wild_q_indices1:
        gt_label_wild.extend(gt_label[i: i+1])
    else:
        gt_label_val.extend(gt_label[i: i+1])
gt_label_test = np.asarray(gt_label_test)
gt_label_wild = np.asarray(gt_label_wild)
gt_label_val = np.asarray(gt_label_val)

In [12]:
wild_q_indices

array([6572, 7314, 4119, ..., 5089, 1247, 7008])

In [13]:
dataset_df["split"] = ["test"] * 7983

In [14]:
dataset_df.loc[wild_q_indices, "split"] = "train"

In [18]:
dataset_df["split"].value_counts()

train    5987
test     1996
Name: split, dtype: int64

In [21]:
dataset_df.loc[wild_q_indices2, "split"] = "val"

In [27]:
dataset_df.to_csv("coqa_haloscope_llama7B.csv")

In [25]:
dataset_df["is_hal"] = gt_label

In [26]:
dataset_df

,context,question,answer,additional_answers,id,generated_answer,split,is_hal
0,"Once upon a time, in a barn near a farm house,...",What color was Cotton?,"{'text': 'white', 'answer_start': 59}","[white, white, white]",3dr23u6we5exclen4th8uq9rb42tel_0,Cotton was a white kitten.,train,0
1,"Once upon a time, in a barn near a farm house,...",Where did she live?,"{'text': 'in a barn', 'answer_start': 18}","[in a barn, in a barn, in a barn near]",3dr23u6we5exclen4th8uq9rb42tel_1,above the barn.,train,0
2,"Once upon a time, in a barn near a farm house,...",She live alone?,"{'text': 'no', 'answer_start': 196}","[no, No, no]",3dr23u6we5exclen4th8uq9rb42tel_2,no.,test,0
3,"Once upon a time, in a barn near a farm house,...",Who did she live with?,"{'text': 'with her mommy and 5 sisters', 'answ...","[her mommy and 5 other sisters, her mommy and ...",3dr23u6we5exclen4th8uq9rb42tel_3,her mommy and 5 other sisters.,test,1
4,"Once upon a time, in a barn near a farm house,...",What color was her sisters?,"{'text': 'orange and white', 'answer_start': 428}","[orange with white tiger stripes, orange, orange]",3dr23u6we5exclen4th8uq9rb42tel_4,orange with white tiger stripes.,test,1
...,...,...,...,...,...,...,...,...
7978,"Las Vegas (, Spanish for ""The Meadows""), offic...","where do the nickname ""Sin City"" come from?",{'text': 'The city's tolerance for numerous fo...,[tolerance for numerous forms of adult enterta...,3ixqg4fa2tygl3tpwwa12i2uf58b90_7,its tolerance for numerous forms of adult ente...,train,1
7979,"Las Vegas (, Spanish for ""The Meadows""), offic...",Which state it in?,"{'text': 'Nevada', 'answer_start': 100}","[Nevada, Nevada,, Nevada]",3ixqg4fa2tygl3tpwwa12i2uf58b90_8,Nevada.,val,1
7980,"Las Vegas (, Spanish for ""The Meadows""), offic...",Is it located in a desert?,"{'text': 'Yes', 'answer_start': 326}","[yes, yes, Yes]",3ixqg4fa2tygl3tpwwa12i2uf58b90_9,Yes.,train,1
7981,"Las Vegas (, Spanish for ""The Meadows""), offic...",what that name of that desert?,"{'text': 'Mojave Desert.', 'answer_start': 345}","[Mojave Desert, Mojave Desert., Mojave]",3ixqg4fa2tygl3tpwwa12i2uf58b90_10,Mojave Desert.,train,1
